In [ ]:
import vitaldb
import pandas as pd
import numpy as np
import os

output_dir = "../patient_raw_data"
os.makedirs(output_dir, exist_ok=True)

# Manual setting for number of cases to download.
# Set an integer (e.g. 100) to limit downloads, or set to None to download all remaining cases.
# Optionally, uncomment below to prompt interactively:
# num_cases_to_download = int(input("Enter number of cases to download: "))
num_cases_to_download = 3000

track_names = [
    "SNUADC/ECG_II",         # Downsampled to 1 Hz
    "SNUADC/PLETH",          # Downsampled to 1 Hz
    "Solar8000/HR",          # Native ~0.5 - 1 Hz
    "Solar8000/ART_SBP",     # Native ~0.5 - 1 Hz
    "Solar8000/ART_DBP",     # Native ~0.5 - 1 Hz
    "Solar8000/ART_MBP",     # Native ~0.5 - 1 Hz
    "Solar8000/PLETH_SPO2",  # Native ~0.5 - 1 Hz
    "Solar8000/RR_CO2",      # Native ~0.5 - 1 Hz
    "Solar8000/ETCO2",       # Native ~0.5 - 1 Hz
    "Primus/FIO2",           # Native ~1 Hz
    "Solar8000/BT"           # Native ~0.5 - 1 Hz
]

In [9]:
# Fetch track catalog from VitalDB
print("Fetching track information from VitalDB...")
df_trks = pd.read_csv("https://api.vitaldb.net/trks")

# Filter cases that contain ALL 11 required tracks
filtered_trks = df_trks[df_trks["tname"].isin(track_names)]
case_track_counts = filtered_trks.groupby("caseid")["tname"].nunique()
candidate_cases = case_track_counts[case_track_counts == len(track_names)].index.values
candidate_cases = np.sort(
    case_track_counts[case_track_counts == len(track_names)].index.values
)

# Identify which candidate cases are already downloaded
existing_files = set(os.listdir(output_dir))
remaining_cases = [c for c in candidate_cases if f"patient_{c}_1hz.csv" not in existing_files]

# Determine cases to download based on manual count setting
if 'num_cases_to_download' in globals() and num_cases_to_download is not None:
    cases_to_download = remaining_cases[:int(num_cases_to_download)]
else:
    cases_to_download = remaining_cases

print(f"Total cases with ALL {len(track_names)} tracks: {len(candidate_cases)}")
print(f"Already downloaded: {len(candidate_cases) - len(remaining_cases)}")
print(f"Remaining available: {len(remaining_cases)}")
print(f"Cases selected for download: {len(cases_to_download)}")

# Download missing patient raw data
for idx, caseid in enumerate(cases_to_download, start=1):
    file_name = f"patient_{caseid}_1hz.csv"
    file_path = os.path.join(output_dir, file_name)
    
    try:
        # Load raw signals sampled at 1 Hz interval
        vals = vitaldb.load_case(caseid, track_names, interval=1)
        if vals is None or len(vals) == 0:
            continue
            
        df = pd.DataFrame(vals, columns=track_names)
        
        # Add timestamp/seconds index
        df.insert(0, "Time_sec", np.arange(len(df)))
        
        # Save patient raw data CSV
        df.to_csv(file_path, index=False)
        
        print(f"[{idx}/{len(cases_to_download)}] Downloaded patient {caseid} -> {file_name}")
            
    except Exception as e:
        print(f"Error downloading patient {caseid}: {e}")

print("Patient raw data extraction complete!")

Fetching track information from VitalDB...
Total cases with ALL 11 tracks: 3352
Already downloaded: 3088
Remaining available: 264
Cases selected for download: 264
[1/264] Downloaded patient 5904 -> patient_5904_1hz.csv
[2/264] Downloaded patient 5907 -> patient_5907_1hz.csv
[3/264] Downloaded patient 5908 -> patient_5908_1hz.csv
[4/264] Downloaded patient 5911 -> patient_5911_1hz.csv
[5/264] Downloaded patient 5912 -> patient_5912_1hz.csv
[6/264] Downloaded patient 5914 -> patient_5914_1hz.csv
[7/264] Downloaded patient 5916 -> patient_5916_1hz.csv


KeyboardInterrupt: 